In [ ]:
import warnings
from pathlib import Path

import torch
import torch.nn as nn

warnings.simplefilter("ignore", UserWarning)
import mltrainer

mltrainer.__version__

Let's get some data.

Same fashion MNIST as in `1_pytorch_intro/06_classification.ipynb`, loaded the same way: the hub
hands over `PIL` images, so a `process` function turns each one into a `(1, 28, 28)` float tensor
and a `collate_fn` stacks a list of rows into a batch. If any of that is unfamiliar, go back to
that notebook — here it is three lines of setup, not the lesson.

In [ ]:
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader

fashion = load_dataset("zalando-datasets/fashion_mnist")


def process(item) -> tuple[torch.Tensor, int]:
    img = np.asarray(item["image"], dtype=np.float32) / 255.0
    return torch.from_numpy(img).unsqueeze(0), item["label"]


def collate_fn(items):
    X, y = zip(*[process(item) for item in items])
    return torch.stack(X), torch.tensor(y)


trainloader = DataLoader(
    fashion["train"], batch_size=64, shuffle=True, collate_fn=collate_fn
)
validloader = DataLoader(
    fashion["test"], batch_size=64, shuffle=False, collate_fn=collate_fn
)

In [ ]:
len(trainloader), len(validloader)

Those are the number of *batches* per epoch, not the number of images. We can obtain one batch:

In [ ]:
x, y = next(iter(trainloader))
x.shape, y.shape

`(64, 1, 28, 28)`: a batch of 64, one channel, 28x28 pixels — the channels-first convention
`(B, C, H, W)`. The label is an integer per image.

Note that this is a rung the text path in lesson 1 never had. A convolution needs the *spatial*
axes intact, so an image item stays `(C, H, W)` instead of collapsing to a `(D)` vector, and the
batch is rank 4 rather than rank 2 (see [tensor-ladder.md](../1_pytorch_intro/tensor-ladder.md)).
Getting back down to something a `Linear` layer can eat is a large part of what this notebook is
about.

Lets pull this through a Conv2d layer:

In [ ]:
in_channels = x.shape[1]

In [ ]:
conv = nn.Conv2d(
    in_channels=in_channels, out_channels=64, kernel_size=3, padding=(1, 1), stride=1
)
out = conv(x)
out.shape

What is happening here? Can you explain all the parameters, and relate them to the outputshape?

Let's see what happens if we change the padding:

In [ ]:
conv = nn.Conv2d(
    in_channels=in_channels,
    out_channels=64,
    kernel_size=3,
    # padding=(0,0),
    stride=1,
)
out = conv(x)
out.shape

And if we change the stride from the default 1 to 2:

In [ ]:
conv = nn.Conv2d(
    in_channels=in_channels, out_channels=64, kernel_size=3, padding=(1, 1), stride=2
)
out = conv(x)
out.shape

As you can see, you need to think about what is going in and out of the convolution. We can stitch multiple layers together like this:

In [ ]:
convolutions = nn.Sequential(
    nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)
out = convolutions(x)
out.shape

As you can see, the dimensions of the featuremap have become really small. You need to take this into account: If we would have started with a smaller image, we could get errors...

In [ ]:
x_too_small = torch.rand((32, 1, 12, 12))

try:
    convolutions(x_too_small)
except RuntimeError as err:
    print("ERROR:", err)

At this point our `out` has 32 activation maps, each 2x2 big.

If we want to pull the activation maps through a neural network (A dense layer) we will need to flatten them (do you understand what happens if you dont do that?)

In [ ]:
input_nn = nn.Flatten()(out)
input_nn.shape

Note that there are potential problems connecting the image layers and the linear layers:
- Conv2d and MaxPool both expect 4 dimensional data (batch, channels/activationmaps, width, height)
- Linear layers expect 2 dimensional data (batch, features)
- Linear layers wont crash if you feed them data with more dimensions! However, they will just work on the last dimension, and thats probably not what you want.

This means we need to somehow transform the 4D data into 2D. There are some options here:
- Some sort of aggregation; the activationmaps are typically small (eg 2x2) and they indicate that the filter has detected a features. There are a lot of different ways to aggregate this: mean, max, min, sum, etc...
- Flatten: a flatten layer simple transforms (batch, C, W, H) into (batch, C * W * H). lets say you have (32, 32, 2, 2) than after a flatten you end up with (32, 128). The problem here is, when you use a different amount of Conv2d layers, or a different stride or padding, you will end up with a different size of activationmap, eg (32, 32, 3, 3), which would mean you would end up with 32 * 3 * 3 = 288 features. 

I have solved this problem by calculating the size of the activationmap with the ._conv_test method. After I calculate the size of the map (eg (2,2)) I can create an AvgPool2d layer that will take the average of the (2,2) map. This way you will always end up with (batch, filters, 1, 1) and after the flatten this will be filter * 1 * 1, which is exactly the amount of filters.

In [ ]:
avgpool = nn.AvgPool2d((2, 2))
pooled = avgpool(out)
pooled.shape

If we flatten this, we obtain 32x1x1 numbers, which is still 32, which makes designing your model a bit easier (and you might also argue that taking the average is a good approach in terms of model logic)

Let's combine it all together, and add a _conv_test method to create the right size for the AvgPool2D layer.

In [ ]:
import torch
from loguru import logger
from torch import nn
from torchinfo import summary


# Define model
class CNN(nn.Module):
    def __init__(self, filters: int, units1: int, units2: int, input_size: tuple):
        super().__init__()
        self.in_channels = input_size[
            1
        ]  # (batch x channels x height x width), so we need the second element
        self.input_size = input_size
        self.filters = filters
        self.units1 = units1
        self.units2 = units2

        self.convolutions = nn.Sequential(
            nn.Conv2d(self.in_channels, filters, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(filters, filters, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        activation_map_size = self._conv_test(self.input_size)
        logger.info(f"Aggregating activationmap with size {activation_map_size}")
        self.agg = nn.AvgPool2d(activation_map_size)

        self.dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, 10),
        )

    def _conv_test(self, input_size):
        x = torch.ones(input_size, dtype=torch.float32)
        x = self.convolutions(x)
        return x.shape[-2:]

    def forward(self, x):
        x = self.convolutions(x)
        x = self.agg(x)
        logits = self.dense(x)
        return logits

In [ ]:
model = CNN(filters=128, units1=128, units2=64, input_size=(32, 3, 224, 224))
summary(model, input_size=(32, 3, 224, 224), device="cpu")

In [ ]:
model = CNN(filters=128, units1=128, units2=64, input_size=(32, 1, 28, 28))
summary(model, input_size=(32, 1, 28, 28), device="cpu")

That summary reports about 322k parameters. You will always need to judge that relative to your input data: 

- how many observations do you have? 
- maybe even more important: how many features do you have? Images sized 28x28 will need much less complexity than images sized 224x224 (note how the first one has 784 features, the second one more than 50.000!)
- Do you think the model needs a lot of complexity, or not so much? E.g. classifying if there is a stamp, or not, on a piece of paper is much easier than classifying the age of a face.

Also think about:
What is the trade off between adding more complexity? Or reducing complexity?

Try to answer this trade of in terms of:

- speed
- generalization
- accuracy

Eg 512 filters might add 0.1 % accuracy, but it might double training time. Is that worth it? Often, not...

We will need to tell the model how good it is performing. To do that, we will need to pick a loss function $\mathcal{L}$. You met these in `1_pytorch_intro/05_losses.ipynb`; for a multi-class problem like this one, `CrossEntropyLoss` is the pick.

In [ ]:
import torch.optim as optim
from mltrainer import Trainer, metrics

optimizer = optim.Adam
loss_fn = torch.nn.CrossEntropyLoss()
accuracy = metrics.Accuracy()

In [ ]:
model = CNN(filters=128, units1=128, units2=64, input_size=(32, 1, 28, 28))

In [ ]:
yhat = model(x)
accuracy(y, yhat)

In [ ]:
log_dir = Path("modellog").resolve()
if not log_dir.exists():
    log_dir.mkdir(parents=True)

We now have everything we need to train the model.

In [ ]:
from mltrainer import ReportTypes, TrainerSettings

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(validloader),
    reporttypes=[ReportTypes.TENSORBOARD],
)
settings

In [ ]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

In [ ]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optimizer,
    traindataloader=trainloader,
    validdataloader=validloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

In [ ]:
trainer.loop()

Have a look at `mltrainer`'s `imagemodels.py`. You will find this model there, but also one built
with a more modular strategy — see the `ConvBlock` and `CNNblocks` architectures.

The model you just built is also saved next to this notebook as `cnn.py`, with `num_classes` as an
argument instead of a hardcoded 10. The next two notebooks import it from there, so the CNN they
compare against a pretrained encoder is the one *you* wrote, not a black box from a library.

You have now seen where a good representation of an image can come from twice: **you design it**
(the `Conv2d` parameters you chose by hand: kernel size, stride, padding, how many layers) and
**you learn it** (the filter weights, trained on 60.000 fashion images).

There is a third answer, and it is the one most of the field actually uses: **you take one that
already exists**. That is `02_pretrained.ipynb`.